# Entrenamiento QLoRA controlado

Proyecto 3 · Deep Learning y Sistemas Inteligentes · Sección 30

## Hipótesis registrada antes de entrenar

Aumentar `r` de 16 a 32 puede mejorar consultas medias y difíciles al añadir capacidad para relaciones y composición SQL, a cambio de más parámetros y tiempo. La ganancia puede ser nula si el límite es la calidad de los datos.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "config.json").exists(), "Ejecuta el notebook desde la raíz o notebooks/."
SEED = 2026

## 1. Verificación de entorno T4

In [ ]:
import torch
assert torch.cuda.is_available(), 'Activa GPU T4 en Colab.'
print(torch.cuda.get_device_name(0))

## 2. Confirmar que solo cambia el rango LoRA

In [ ]:
a = json.loads((ROOT / 'configuraciones/qlora_r16.json').read_text())
b = json.loads((ROOT / 'configuraciones/qlora_r32.json').read_text())
differences = {key: (a.get(key), b.get(key)) for key in sorted(a | b) if a.get(key) != b.get(key)}
assert set(differences) <= {'nombre', 'r'}
differences

## 3. Smoke test

Ejecuta primero dos pasos por configuración. Si ambos terminan, quita `--smoke` para las corridas reales.

In [ ]:
RUN_TRAINING = False  # Cambiar a True únicamente en Colab T4.
if RUN_TRAINING:
    for name in ('qlora_r16', 'qlora_r32'):
        subprocess.run([sys.executable, str(ROOT / 'scripts/train_qlora.py'), '--config', str(ROOT / f'configuraciones/{name}.json'), '--output', str(ROOT / f'resultados/adapters/{name}'), '--smoke'], check=True)
else:
    print('Espacio preparado. Activa RUN_TRAINING en T4.')

## 4. Corridas completas

In [ ]:
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    for name in ('qlora_r16', 'qlora_r32'):
        subprocess.run([sys.executable, str(ROOT / 'scripts/train_qlora.py'), '--config', str(ROOT / f'configuraciones/{name}.json'), '--output', str(ROOT / f'resultados/adapters/{name}')], check=True)

## Artefactos esperados

Cada directorio contiene adapter, tokenizer, `run_metrics.json` y `log_history.json`. No selecciones por loss: la decisión final se toma por EX ponderada en el notebook 03.